# ScreenGuide: Qwen3.5-4B (via local-jev) on the held-out benchmark

**Before you start:** Runtime → Change runtime type → **GPU** (T4 works). Then **Run all**.

What this measures: **accuracy** of Qwen3.5-4B, served by [local-jev](https://github.com/amithgc/local-jev), on the same 35 held-out goals and the full-tournament validation set. Accuracy is the same model on any hardware. **Latency here is the Colab GPU's, not the Mac's.** Measure Mac latency separately.

Running it here only because the 9 GB download is too slow on the dev machine's connection.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /content
!rm -rf app_builders_hack local-jev
!git clone -q https://github.com/EZA2405/app_builders_hack
!git clone -q --depth 1 https://github.com/amithgc/local-jev
!pip -q install -e /content/local-jev
!python -c "import transformers, torch; print('transformers', transformers.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
# Download the weights first (fast on Colab), then start the server.
!hf download Qwen/Qwen3.5-4B --quiet && echo downloaded
import subprocess, time, urllib.request
server = subprocess.Popen(["local-jev", "serve", "--model", "llm-qwen3.5-4b", "--host", "127.0.0.1", "--port", "8765"],
                          stdout=open("/content/localjev.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8765/healthz", timeout=2); break
    except Exception:
        time.sleep(2)
print("server up"); print(open("/content/localjev.log").read()[-800:])

In [ ]:
%cd /content/app_builders_hack/ml
import re, subprocess, glob
def bench(fixtures, label, group):
    hits = total = 0
    for f in fixtures:
        out = subprocess.run(["python3", "bench_localjev.py", "--url", "http://127.0.0.1:8765", "--model", "llm-qwen3.5-4b",
                              "--group", str(group), "--fixture", f], capture_output=True, text=True).stdout
        print(out)
        m = re.search(r"accuracy (\d+)/(\d+)", out)
        if m: hits += int(m[1]); total += int(m[2])
    print(f"===== Qwen3.5-4B {label} (group {group}): {hits}/{total} =====")
    return hits, total

val = bench(sorted(glob.glob("fixtures/val/*.json")), "full-tournament VAL", 200)

In [ ]:
held = bench([f"fixtures/{a}_real.json" for a in ["preview", "finder", "systemsettings", "safari"]], "HELD-OUT 35", 200)
print("VAL", val, "| HELD-OUT", held)

Post both numbers in the team chat. Add them to `ml/RESULTS.md` as **Qwen3.5-4B via local-jev**, noting that latency is from the Colab GPU.